In [ ]:
# 🧱 Cell 1: Upload ZIP file(s) that contain your CSV data

from google.colab import files
import os

UPLOAD_DIR = '/content/uploads'
os.makedirs(UPLOAD_DIR, exist_ok=True)

print("📦 لطفاً فایل(های) ZIP دیتاتون رو انتخاب کنید (می‌تونید چندتا هم‌زمان انتخاب کنید).")
uploaded = files.upload()  # باز کردن دیالوگ آپلود

zip_paths = []
for fname, data in uploaded.items():
    out_path = os.path.join(UPLOAD_DIR, fname)
    with open(out_path, 'wb') as f:
        f.write(data)
    if fname.lower().endswith('.zip'):
        zip_paths.append(out_path)

print(f"\n✅ آپلود کامل شد. تعداد ZIP: {len(zip_paths)}")
for p in zip_paths:
    print(" -", p)

if not zip_paths:
    raise SystemExit("⛔ هیچ فایل .zip آپلود نشد. لطفاً حداقل یک ZIP شامل CSV آپلود کنید.")


📦 لطفاً فایل(های) ZIP دیتاتون رو انتخاب کنید (می‌تونید چندتا هم‌زمان انتخاب کنید).


Saving merged_all_parts.zip to merged_all_parts.zip

✅ آپلود کامل شد. تعداد ZIP: 1
 - /content/uploads/merged_all_parts.zip


In [ ]:
# 🧱 Cell 2: Extract all ZIPs and list CSV files (run after Cell 1)

import zipfile, glob, os

DATA_DIR = '/content/data'
os.makedirs(DATA_DIR, exist_ok=True)

csv_paths = []

for zp in zip_paths:
    print(f"🔎 در حال Extract: {zp}")
    with zipfile.ZipFile(zp, 'r') as zf:
        for member in zf.infolist():
            name = member.filename.replace('\\','/')
            # جلوگیری از path traversal
            if os.path.isabs(name) or '..' in name:
                continue
            zf.extract(member, DATA_DIR)
    print("   ✅ انجام شد.")

# پیداکردن CSVها (case-insensitive)
for ext in ('*.csv','*.CSV'):
    csv_paths.extend(glob.glob(os.path.join(DATA_DIR, '**', ext), recursive=True))

if not csv_paths:
    raise SystemExit("⛔ هیچ فایل CSV داخل ZIPها پیدا نشد.")

# مرتب‌سازی بر اساس اندازه (بزرگ‌ترین اول)
csv_paths = sorted(csv_paths, key=lambda p: os.path.getsize(p), reverse=True)

print(f"\n📄 تعداد CSV پیدا شده: {len(csv_paths)}")
for i, p in enumerate(csv_paths[:10], 1):
    size_mb = os.path.getsize(p)/1024/1024
    print(f"{i:>2}. {p}  ({size_mb:.2f} MB)")

# انتخاب خودکار بزرگ‌ترین CSV به عنوان فایل اصلی
CSV_PATH = csv_paths[0]
print("\n➡️ انتخاب خودکار بزرگ‌ترین CSV به عنوان فایل اصلی:")
print("CSV_PATH =", CSV_PATH)


🔎 در حال Extract: /content/uploads/merged_all_parts.zip
   ✅ انجام شد.

📄 تعداد CSV پیدا شده: 1
 1. /content/data/merged_all_parts.csv  (584.29 MB)

➡️ انتخاب خودکار بزرگ‌ترین CSV به عنوان فایل اصلی:
CSV_PATH = /content/data/merged_all_parts.csv


In [ ]:
# 🧱 Cell 3: Safe preview (head & tail) + columns

import os, io

import pandas as pd

# اگر از قبل CSV_PATH در سلول 2 تنظیم شده، از همان استفاده می‌کنیم
try:
    CSV_PATH
except NameError:
    raise SystemExit("⛔ CSV_PATH تعریف نشده. اول سلول 1 و 2 را اجرا کنید.")

size_mb = os.path.getsize(CSV_PATH)/1024/1024
print(f"📄 فایل انتخاب‌شده:\n{CSV_PATH}\nحجم تقریبی: {size_mb:.2f} MB")

# --- Head: فقط 5 ردیف اول ---
try:
    df_head = pd.read_csv(CSV_PATH, nrows=5)
except Exception as e:
    print("⚠️ خطا در خواندن head با pandas:", e)
    df_head = None

if df_head is not None:
    print("\n🔹 5 ردیف اول:")
    display(df_head)

# --- Columns: از همان df_head استخراج می‌کنیم (اگر ممکن نبود، یک خواندن سریع فقط برای header) ---
if df_head is not None:
    columns = list(df_head.columns)
else:
    with open(CSV_PATH, 'rb') as f:
        header = f.readline().decode('utf-8', errors='replace').rstrip('\n\r')
    columns = header.split(',')
print(f"\n🔧 ستون‌ها ({len(columns)}):")
print(columns)

# --- Tail: 5 ردیف آخر بدون لود کامل ---
def read_tail_lines(path, max_bytes=1024*64, n_lines=5, encoding='utf-8'):
    """خواندن n_lines خط آخر فایل متنی با عقب‌گرد از انتهای فایل."""
    sz = os.path.getsize(path)
    read_bytes = min(max_bytes, sz)
    with open(path, 'rb') as f:
        f.seek(sz - read_bytes)
        chunk = f.read(read_bytes)
    text = chunk.decode(encoding, errors='replace')
    lines = text.splitlines()
    # ممکن است اولین خط ناقص باشد؛ ما فقط خطوط کامل انتهایی را می‌گیریم
    last_lines = lines[-n_lines:]
    return last_lines

try:
    tail_lines = read_tail_lines(CSV_PATH, max_bytes=1024*128, n_lines=5)
    print("\n🔹 5 خط آخر (خام):")
    for i, line in enumerate(tail_lines, 1):
        print(f"{i:>2}: {line[:300]}")
    # تلاش برای parse این 5 خط برای نمایش مرتب
    tail_csv = "\n".join([",".join(columns)] + tail_lines)
    df_tail = pd.read_csv(io.StringIO(tail_csv))
    print("\n🔹 5 ردیف آخر (Parse شده):")
    display(df_tail)
except Exception as e:
    print("⚠️ نشد tail را parse کنیم؛ نمایش خام کافیست. جزئیات:", e)


📄 فایل انتخاب‌شده:
/content/data/merged_all_parts.csv
حجم تقریبی: 584.29 MB

🔹 5 ردیف اول:


,@timestamp,source.address,destination.address,destination.port,suricata.eve.alert.signature,message,event.type,http.response.status_code,rule.category,rule.name,url.original,user_agent.original,suricata.eve.alert.metadata,source_file
0,"Sep 28, 2025 @ 10:28:02.709",10.10.10.200,102.49.30.47,"16,394",ET INFO Session Traversal Utilities for NAT (S...,Misc activity,allowed,-,Misc activity,ET INFO Session Traversal Utilities for NAT (S...,-,-,"{\n ""confidence"": ""High"",\n ""signature_sever...",Alerts1.csv
1,"Sep 28, 2025 @ 10:28:02.709",10.10.10.200,102.49.30.47,"16,394",ET INFO Session Traversal Utilities for NAT (S...,Misc activity,allowed,-,Misc activity,ET INFO Session Traversal Utilities for NAT (S...,-,-,"{\n ""confidence"": ""High"",\n ""signature_sever...",Alerts1.csv
2,"Sep 28, 2025 @ 10:28:01.706",10.10.10.200,102.49.30.47,"16,394",ET INFO Session Traversal Utilities for NAT (S...,Misc activity,allowed,-,Misc activity,ET INFO Session Traversal Utilities for NAT (S...,-,-,"{\n ""confidence"": ""High"",\n ""signature_sever...",Alerts1.csv
3,"Sep 28, 2025 @ 10:28:01.706",10.10.10.200,102.49.30.47,"16,394",ET INFO Session Traversal Utilities for NAT (S...,Misc activity,allowed,-,Misc activity,ET INFO Session Traversal Utilities for NAT (S...,-,-,"{\n ""confidence"": ""High"",\n ""signature_sever...",Alerts1.csv
4,"Sep 28, 2025 @ 10:28:00.707",10.10.10.200,102.49.30.47,"16,394",ET INFO Session Traversal Utilities for NAT (S...,Misc activity,allowed,-,Misc activity,ET INFO Session Traversal Utilities for NAT (S...,-,-,"{\n ""confidence"": ""High"",\n ""signature_sever...",Alerts1.csv



🔧 ستون‌ها (14):
['@timestamp', 'source.address', 'destination.address', 'destination.port', 'suricata.eve.alert.signature', 'message', 'event.type', 'http.response.status_code', 'rule.category', 'rule.name', 'url.original', 'user_agent.original', 'suricata.eve.alert.metadata', 'source_file']

🔹 5 خط آخر (خام):
 1: "Jul 22, 2025 @ 08:09:01.582",192.168.20.13,172.16.60.2,"8,080",-,-,"connection, end",-,-,-,-,-,-,Suricata-table.csv
 2: "Jul 22, 2025 @ 08:09:01.582",192.168.20.15,172.16.60.2,"8,080",-,-,"connection, end",-,-,-,-,-,-,Suricata-table.csv
 3: "Jul 22, 2025 @ 08:09:01.582",192.168.10.11,172.16.60.2,"8,080",-,-,"connection, end",-,-,-,-,-,-,Suricata-table.csv
 4: "Jul 22, 2025 @ 08:09:01.582",192.168.10.11,172.16.60.2,"8,080",-,-,"connection, end",-,-,-,-,-,-,Suricata-table.csv
 5: "Jul 22, 2025 @ 08:09:01.538",172.16.60.2,10.10.10.69,"4,545",-,-,-,200,-,-,http://149.154.167.91:80/api,Mozilla/5.0,-,Suricata-table.csv

🔹 5 ردیف آخر (Parse شده):


,@timestamp,source.address,destination.address,destination.port,suricata.eve.alert.signature,message,event.type,http.response.status_code,rule.category,rule.name,url.original,user_agent.original,suricata.eve.alert.metadata,source_file
0,"Jul 22, 2025 @ 08:09:01.582",192.168.20.13,172.16.60.2,"8,080",-,-,"connection, end",-,-,-,-,-,-,Suricata-table.csv
1,"Jul 22, 2025 @ 08:09:01.582",192.168.20.15,172.16.60.2,"8,080",-,-,"connection, end",-,-,-,-,-,-,Suricata-table.csv
2,"Jul 22, 2025 @ 08:09:01.582",192.168.10.11,172.16.60.2,"8,080",-,-,"connection, end",-,-,-,-,-,-,Suricata-table.csv
3,"Jul 22, 2025 @ 08:09:01.582",192.168.10.11,172.16.60.2,"8,080",-,-,"connection, end",-,-,-,-,-,-,Suricata-table.csv
4,"Jul 22, 2025 @ 08:09:01.538",172.16.60.2,10.10.10.69,"4,545",-,-,-,200,-,-,http://149.154.167.91:80/api,Mozilla/5.0,-,Suricata-table.csv


In [ ]:
# 🧱 Cell 4: Lightweight dtype inference by sampling small chunks

import pandas as pd
from collections import Counter

SAMPLE_TARGET = 20000         # مجموع ردیف‌های نمونه‌گیری‌شده
CHUNK_SIZE     = 5000          # اندازه‌ی هر چانک
MAX_CHUNKS     = max(1, SAMPLE_TARGET // CHUNK_SIZE)

# اگر برخی ستون‌ها زمان هستند، نام‌های رایج را اینجا بگذاریم تا تست تبدیل datetime بدهیم
LIKELY_TIME_COLS = {'@timestamp','timestamp','time','event_time','flow_start','flow_end','ts'}

sampled = []
rows_collected = 0
for i, chunk in enumerate(pd.read_csv(CSV_PATH, chunksize=CHUNK_SIZE), 1):
    sampled.append(chunk)
    rows_collected += len(chunk)
    if i >= MAX_CHUNKS or rows_collected >= SAMPLE_TARGET:
        break

if not sampled:
    raise SystemExit("⛔ نتوانستیم نمونه‌ای بخوانیم.")

sample_df = pd.concat(sampled, ignore_index=True)
print(f"✅ نمونه‌گیری انجام شد: {len(sample_df):,} ردیف")

# تابع تشخیص نوع ساده
def guess_dtype(series: pd.Series):
    s = series.dropna()
    if s.empty:
        return 'string'
    # boolean-like
    lower_vals = set(map(lambda x: str(x).strip().lower(), s.sample(min(1000, len(s)), random_state=42)))
    if lower_vals.issubset({'true','false','0','1','yes','no'}):
        return 'boolean'
    # integer?
    try:
        s_int = pd.to_numeric(s, errors='raise', downcast='integer')
        if (s_int % 1 == 0).all():
            return 'int64'
    except:
        pass
    # float?
    try:
        pd.to_numeric(s, errors='raise', downcast='float')
        return 'float64'
    except:
        pass
    # datetime candidate?
    if series.name and series.name.lower() in LIKELY_TIME_COLS:
        try:
            pd.to_datetime(s, errors='raise')
            return 'datetime64[ns]'
        except:
            pass
    # strings (categorical?) — اگر کاردینالیتی کم باشد، بعداً می‌توانیم category کنیم
    return 'string'

dtypes_guess = {}
cardinality = {}
for col in sample_df.columns:
    g = guess_dtype(sample_df[col])
    dtypes_guess[col] = g
    # کاردینالیتی سریع
    try:
        cardinality[col] = sample_df[col].nunique(dropna=True)
    except:
        cardinality[col] = None

# پیشنهاد dtypeهای نهایی برای خواندن اصلی
DTYPES = {}
PARSE_DATES = []

for col, g in dtypes_guess.items():
    if g == 'int64':
        DTYPES[col] = 'Int64'   # نوع nullable در pandas
    elif g == 'float64':
        DTYPES[col] = 'float32' # کم‌حافظه‌تر
    elif g == 'boolean':
        DTYPES[col] = 'boolean' # nullable boolean
    elif g == 'datetime64[ns]':
        PARSE_DATES.append(col)
    else:
        # رشته‌ها: اگر کاردینالیتی کم باشد می‌توانیم category کنیم (در خواندن اصلی قابل اعمال است)
        DTYPES[col] = 'string'

print("\n🧭 حدس نوع داده‌ها:")
for col in sample_df.columns:
    print(f"{col:>20}  ->  {dtypes_guess[col]:<14}  (unique≈{cardinality.get(col)})")

print("\n📦 پیشنهاد برای read_csv:")
print(" - PARSE_DATES =", PARSE_DATES)
print(" - DTYPES (نمونه):")
for i, (k,v) in enumerate(DTYPES.items()):
    if i>=10:
        print("   ...")
        break
    print(f"   {k}: {v}")

# نمایش چند آمار سبک از نمونه
print("\n📊 چند آمار سبک از نمونه:")
try:
    print(sample_df.describe(include='all').transpose().head(15))
except Exception as e:
    print("⚠️ نتوانستیم describe کامل بگیریم:", e)


✅ نمونه‌گیری انجام شد: 20,000 ردیف

🧭 حدس نوع داده‌ها:
          @timestamp  ->  string          (unique≈10356)
      source.address  ->  string          (unique≈134)
 destination.address  ->  string          (unique≈670)
    destination.port  ->  string          (unique≈398)
suricata.eve.alert.signature  ->  string          (unique≈87)
             message  ->  string          (unique≈12)
          event.type  ->  string          (unique≈1)
http.response.status_code  ->  string          (unique≈11)
       rule.category  ->  string          (unique≈12)
           rule.name  ->  string          (unique≈87)
        url.original  ->  string          (unique≈566)
 user_agent.original  ->  string          (unique≈22)
suricata.eve.alert.metadata  ->  string          (unique≈48)
         source_file  ->  string          (unique≈1)

📦 پیشنهاد برای read_csv:
 - PARSE_DATES = []
 - DTYPES (نمونه):
   @timestamp: string
   source.address: string
   destination.address: string
   destination.port:

/tmp/ipython-input-3354454679.py:52: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(s, errors='raise')


                              count unique  \
@timestamp                    20000  10356   
source.address                20000    134   
destination.address           20000    670   
destination.port              20000    398   
suricata.eve.alert.signature  20000     87   
message                       20000     12   
event.type                    20000      1   
http.response.status_code     20000     11   
rule.category                 20000     12   
rule.name                     20000     87   
url.original                  20000    566   
user_agent.original           20000     22   
suricata.eve.alert.metadata   20000     48   
source_file                   20000      1   

                                                                            top  \
@timestamp                                          Sep 27, 2025 @ 03:49:56.286   
source.address                                                    192.168.100.5   
destination.address                                         

In [ ]:
# 🧱 Cell 5 (fixed): Chunked read + clean + save Parquet once

import pandas as pd
import numpy as np

CHUNK_SIZE = 200_000   # بسته به RAM تغییر بده
PARQUET_OUT = "/content/data/cleaned_data.parquet"
WRITE_PARQUET = True

total_rows = 0
all_chunks = []

for i, chunk in enumerate(pd.read_csv(CSV_PATH, chunksize=CHUNK_SIZE), 1):
    # --- پاکسازی پایه ---
    # timestamp
    chunk["@timestamp"] = pd.to_datetime(
        chunk["@timestamp"], errors="coerce", format="%b %d, %Y @ %H:%M:%S.%f"
    )

    # destination.port → عدد
    chunk["destination.port"] = pd.to_numeric(
        chunk["destination.port"].astype(str).str.replace(",","", regex=False), errors="coerce"
    )

    # http.response.status_code → عدد
    chunk["http.response.status_code"] = pd.to_numeric(
        chunk["http.response.status_code"], errors="coerce"
    )

    # "-" → NaN
    chunk = chunk.replace("-", np.nan)

    total_rows += len(chunk)
    all_chunks.append(chunk)

    print(f"✅ چانک {i} پردازش شد ({len(chunk):,} ردیف). مجموع تا الان: {total_rows:,}")

# حالا merge
df = pd.concat(all_chunks, ignore_index=True)
print(f"\n📊 مجموع ردیف‌ها: {len(df):,}")

if WRITE_PARQUET:
    df.to_parquet(PARQUET_OUT, engine="pyarrow", index=False)
    print(f"📂 ذخیره شد در Parquet: {PARQUET_OUT}")


✅ چانک 1 پردازش شد (200,000 ردیف). مجموع تا الان: 200,000
✅ چانک 2 پردازش شد (200,000 ردیف). مجموع تا الان: 400,000
✅ چانک 3 پردازش شد (200,000 ردیف). مجموع تا الان: 600,000


/tmp/ipython-input-1519873464.py:31: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  chunk = chunk.replace("-", np.nan)


✅ چانک 4 پردازش شد (200,000 ردیف). مجموع تا الان: 800,000


/tmp/ipython-input-1519873464.py:31: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  chunk = chunk.replace("-", np.nan)


✅ چانک 5 پردازش شد (200,000 ردیف). مجموع تا الان: 1,000,000


...

✅ چانک 19 پردازش شد (200,000 ردیف). مجموع تا الان: 3,800,000


/tmp/ipython-input-1519873464.py:31: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  chunk = chunk.replace("-", np.nan)


✅ چانک 20 پردازش شد (200,000 ردیف). مجموع تا الان: 4,000,000


/tmp/ipython-input-1519873464.py:31: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  chunk = chunk.replace("-", np.nan)


✅ چانک 21 پردازش شد (46,428 ردیف). مجموع تا الان: 4,046,428

📊 مجموع ردیف‌ها: 4,046,428
📂 ذخیره شد در Parquet: /content/data/cleaned_data.parquet


In [ ]:
# Cell 6: Chunked feature engineering -> features.parquet (Suricata logs)
# - Lightweight numeric features for unsupervised models (IsolationForest etc.)
# - Chunked processing to avoid OOM on Colab Free
# - Robust schema handling for Parquet (no boolean fillna issues, stable dtypes)

import os
import re
import math
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from urllib.parse import urlparse

# =====================
# Configuration
# =====================
try:
    CSV_PATH
    SOURCE = "csv"   # read original CSV in chunks
except NameError:
    SOURCE = "parquet"  # fallback to parquet if CSV_PATH isn't defined

PARQUET_IN  = "/content/data/cleaned_data.parquet"  # produced in Cell 5
FEATURES_OUT = "/content/data/features.parquet"
CHUNK_SIZE   = 200_000

NEEDED_COLS = [
    '@timestamp',
    'source.address','destination.address','destination.port',
    'http.response.status_code','url.original','user_agent.original',
    'suricata.eve.alert.signature','rule.name','rule.category',
    'suricata.eve.alert.metadata','message','event.type'
]

# Remove previous output to avoid schema collisions
try:
    if os.path.exists(FEATURES_OUT):
        os.remove(FEATURES_OUT)
except Exception:
    pass

# =====================
# Helper functions
# =====================

def is_private_ip(ip: str) -> int:
    if not isinstance(ip, str):
        return 0
    if ip.startswith("10.") or ip.startswith("192.168."):
        return 1
    if ip.startswith("172."):
        try:
            sec = int(ip.split(".")[1])
            return 1 if 16 <= sec <= 31 else 0
        except Exception:
            return 0
    return 0


def ip_to_24(ip: str) -> float:
    try:
        o = ip.split(".")
        return float(int(o[0]) * 256 * 256 + int(o[1]) * 256 + int(o[2]))
    except Exception:
        return np.nan


def safe_len(x) -> int:
    try:
        return len(str(x))
    except Exception:
        return 0


def char_entropy(s: str) -> float:
    if not isinstance(s, str) or not s:
        return 0.0
    from collections import Counter
    c = Counter(s)
    n = sum(c.values())
    ps = [v / n for v in c.values()]
    return float(-sum(p * math.log2(p) for p in ps))


def count_chars(s: str, chars="/?&=") -> int:
    if not isinstance(s, str):
        return 0
    return sum(s.count(ch) for ch in chars)


def url_is_host_ip(u: str) -> int:
    try:
        host = urlparse(u).hostname
        if not host:
            return 0
        return 1 if re.fullmatch(r"\d{1,3}(\.\d{1,3}){3}", host) else 0
    except Exception:
        return 0


def ua_flags(ua: str) -> pd.Series:
    ua = (ua or "").lower()
    return pd.Series({
        "ua_len": safe_len(ua),
        "ua_is_browser": int(any(k in ua for k in ["chrome","safari","firefox","edge"])),
        "ua_is_lib": int(any(k in ua for k in ["curl","python-requests","wget","go-http-client"])),
        "ua_is_mobile": int(any(k in ua for k in ["android","iphone","ipad"]))
    })


def build_features(df: pd.DataFrame, row_id_offset: int) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)

    # --- Time -> epoch_ms as float64 (robust to NaT via NaN) ---
    ts = pd.to_datetime(df["@timestamp"], errors="coerce", format="%b %d, %Y @ %H:%M:%S.%f")
    # Use view to get ns, then mask NaT to NaN, then convert to ms float
    ns = ts.view("int64")
    mask_nat = ts.isna()
    ns = ns.astype("float64")  # convert to float to allow NaN
    ns[mask_nat] = np.nan
    out["epoch_ms"] = (ns / 1_000_000.0).astype("float64")
    out["hour"] = ts.dt.hour.astype("Int16")
    out["dow"]  = ts.dt.dayofweek.astype("Int16")

    # --- IPs ---
    out["src_priv"] = df["source.address"].apply(is_private_ip).astype("Int8")
    out["dst_priv"] = df["destination.address"].apply(is_private_ip).astype("Int8")
    out["src_24"]   = df["source.address"].apply(ip_to_24).astype("float32")
    out["dst_24"]   = df["destination.address"].apply(ip_to_24).astype("float32")
    out["src_eq_dst"] = (df["source.address"] == df["destination.address"]).astype("Int8")

    # --- Port / HTTP ---
    ports_clean = df["destination.port"].astype(str).str.replace(",","", regex=False)
    out["dst_port"] = pd.to_numeric(ports_clean, errors="coerce").astype("float64")
    out["port_wellknown"] = (out["dst_port"] <= 1024).fillna(False).astype("Int8")
    out["port_ephemeral"] = (out["dst_port"] >= 49152).fillna(False).astype("Int8")

    out["http_status"] = pd.to_numeric(df["http.response.status_code"], errors="coerce").astype("float64")

    # --- URL ---
    url = df["url.original"].astype("string")
    out["url_len"]      = url.apply(safe_len).astype("Int32")
    out["url_marks"]    = url.apply(count_chars).astype("Int16")
    out["url_entropy"]  = url.apply(char_entropy).astype("float32")
    out["url_host_is_ip"] = url.apply(url_is_host_ip).astype("Int8")

    # --- User-Agent ---
    ua = df["user_agent.original"].astype("string")
    ua_feat = ua.apply(lambda s: ua_flags(s))
    out = pd.concat([out, ua_feat], axis=1)
    out["ua_len"]        = out["ua_len"].astype("Int16")
    out["ua_is_browser"] = out["ua_is_browser"].astype("Int8")
    out["ua_is_lib"]     = out["ua_is_lib"].astype("Int8")
    out["ua_is_mobile"]  = out["ua_is_mobile"].astype("Int8")

    # --- Suricata text fields: length + missing flag (as Int8, not boolean) ---
    for col in [
        "suricata.eve.alert.signature","rule.name","rule.category",
        "suricata.eve.alert.metadata","message","event.type"
    ]:
        s = df[col].astype("string")
        out[col.replace(".", "_") + "_len"]     = s.apply(safe_len).astype("Int16")
        out[col.replace(".", "_") + "_missing"] = s.isna().astype("Int8")

    # --- Row ID (set at the end to avoid any dtype surprises) ---
    n = len(df)
    out["row_id"] = pd.Series(np.arange(n, dtype=np.int64) + row_id_offset, index=df.index, dtype="Int64")
    # Keep row_id as pandas nullable Int64 for schema stability

    return out

# =====================
# Writer loop (chunked, stable schema)
# =====================
writer = None
schema = None
row_id_offset = 0
total_rows = 0

if SOURCE == "csv":
    reader = pd.read_csv(CSV_PATH, chunksize=CHUNK_SIZE)
else:
    import pyarrow.dataset as ds
    dataset = ds.dataset(PARQUET_IN, format="parquet")
    reader = (b.to_pandas() for b in dataset.to_batches(batch_size=CHUNK_SIZE))

for i, chunk in enumerate(reader, 1):
    # Ensure required columns exist
    for c in NEEDED_COLS:
        if c not in chunk.columns:
            chunk[c] = np.nan

    feats = build_features(chunk, row_id_offset=row_id_offset)
    row_id_offset += len(chunk)
    total_rows += len(chunk)

    # Convert to Arrow table
    table = pa.Table.from_pandas(feats, preserve_index=False)
    if writer is None:
        schema = table.schema
        writer = pq.ParquetWriter(FEATURES_OUT, schema)
    else:
        if table.schema != schema:
            table = table.cast(schema)
    writer.write_table(table)

    print(f"🧪 Chunk {i} -> features: {len(chunk):,} rows — total: {total_rows:,}")

if writer is not None:
    writer.close()

print(f"\n✅ Done. Features written to: {FEATURES_OUT}")
print(f"📊 Total rows: {total_rows:,}")


/tmp/ipython-input-275645751.py:118: FutureWarning: Series.view is deprecated and will be removed in a future version. Use ``astype`` as an alternative to change the dtype.
  ns = ts.view("int64")


🧪 Chunk 1 -> features: 200,000 rows — total: 200,000


/tmp/ipython-input-275645751.py:118: FutureWarning: Series.view is deprecated and will be removed in a future version. Use ``astype`` as an alternative to change the dtype.
  ns = ts.view("int64")


🧪 Chunk 2 -> features: 200,000 rows — total: 400,000


/tmp/ipython-input-275645751.py:118: FutureWarning: Series.view is deprecated and will be removed in a future version. Use ``astype`` as an alternative to change the dtype.
  ns = ts.view("int64")


...

🧪 Chunk 19 -> features: 200,000 rows — total: 3,800,000


/tmp/ipython-input-275645751.py:118: FutureWarning: Series.view is deprecated and will be removed in a future version. Use ``astype`` as an alternative to change the dtype.
  ns = ts.view("int64")


🧪 Chunk 20 -> features: 200,000 rows — total: 4,000,000


/tmp/ipython-input-275645751.py:118: FutureWarning: Series.view is deprecated and will be removed in a future version. Use ``astype`` as an alternative to change the dtype.
  ns = ts.view("int64")


🧪 Chunk 21 -> features: 46,428 rows — total: 4,046,428

✅ Done. Features written to: /content/data/features.parquet
📊 Total rows: 4,046,428


In [ ]:
# Cell 7: Sample & fit RobustScaler (clean)
# وظیفه: از features.parquet نمونه می‌گیریم، لیست فیچرها (همه به‌جز row_id) را می‌سازیم و RobustScaler را فیت می‌کنیم.
# خروجی: robust_scaler.joblib و feature_list.json

import json
import pandas as pd
import pyarrow.dataset as ds
import joblib
from sklearn.preprocessing import RobustScaler

FEATURES_IN = "/content/data/features.parquet"
SCALER_OUT = "/content/data/robust_scaler.joblib"
FEATURE_LIST_JSON = "/content/data/feature_list.json"

SAMPLE_FOR_SCALER = 1_000_000   # بر اساس RAM قابل تنظیم
BATCH_SIZE = 150_000            # اندازهٔ هر batch خواندن

# 1) بارگذاری دیتاستِ فیچرها
feat_ds = ds.dataset(FEATURES_IN, format="parquet")
all_cols = feat_ds.schema.names
feature_cols = [c for c in all_cols if c != "row_id"]

# 2) نمونه‌گیری مرحله‌ای تا رسیدن به سقف تعیین‌شده
X_parts = []
rows_collected = 0
for batch in feat_ds.to_batches(batch_size=BATCH_SIZE):
    dfb = batch.to_pandas()
    X_parts.append(dfb[feature_cols])
    rows_collected += len(dfb)
    if rows_collected >= SAMPLE_FOR_SCALER:
        break

if not X_parts:
    raise SystemExit("⛔ features.parquet خالی است یا batchی برنگشت. ابتدا Cell 6 را بساز/اجرا کن.")

X_sample = pd.concat(X_parts, ignore_index=True)
print(f"Sample for scaler: {len(X_sample):,} rows, {len(feature_cols)} features")

# 3) پرکردن NaN (RobustScaler با NaN مشکل دارد)
X_sample = X_sample.fillna(0)

# 4) فیت اسکالر مقاوم به دُم‌های سنگین
scaler = RobustScaler(with_centering=True, with_scaling=True, quantile_range=(10.0, 90.0))
scaler.fit(X_sample.values)

# 5) ذخیرهٔ اسکالر و لیست فیچرها
joblib.dump(scaler, SCALER_OUT)
with open(FEATURE_LIST_JSON, "w") as f:
    json.dump({"features": feature_cols}, f)

print(f"✅ Scaler saved to: {SCALER_OUT}")
print(f"✅ Feature list saved to: {FEATURE_LIST_JSON}")


Sample for scaler: 1,000,000 rows, 32 features
✅ Scaler saved to: /content/data/robust_scaler.joblib
✅ Feature list saved to: /content/data/feature_list.json


In [ ]:
# Cell 8: Train IsolationForest (max-perf, Colab-safe)
# هدف: بیشترین حجم مفیدِ دادهٔ آموزش روی Colab رایگان، با تنظیمات بهینه برای دقت/پایداری
# خروجی: /content/data/iforest_model.joblib

import json
import gc
import joblib
import numpy as np
import pandas as pd
import pyarrow.dataset as ds
from sklearn.ensemble import IsolationForest

FEATURES_IN = "/content/data/features.parquet"
SCALER_IN   = "/content/data/robust_scaler.joblib"
FEATURE_LIST_JSON = "/content/data/feature_list.json"
MODEL_OUT   = "/content/data/iforest_model.joblib"

# ❗ اگر RAM کم آوردی، این عدد را پایین‌تر بگذار (مثلاً 1_500_000)
SAMPLE_FOR_TRAIN = 2_500_000   # حدوداً تا ~2.5M ردیف (32 فیچر) روی Colab رایگان OK است
BATCH_SIZE_TRAIN = 200_000     # سایز batch خواندن از پارکت

# 1) بارگذاری اسکالر و لیست فیچرها
scaler = joblib.load(SCALER_IN)
with open(FEATURE_LIST_JSON, "r") as f:
    feature_cols = json.load(f)["features"]

# 2) جمع‌آوری دادهٔ آموزش مرحله‌ای تا سقف تعیین‌شده
feat_ds = ds.dataset(FEATURES_IN, format="parquet")
train_parts = []
rows_collected = 0
for batch in feat_ds.to_batches(batch_size=BATCH_SIZE_TRAIN):
    dfb = batch.to_pandas()
    train_parts.append(dfb[feature_cols])
    rows_collected += len(dfb)
    if rows_collected >= SAMPLE_FOR_TRAIN:
        break

if not train_parts:
    raise SystemExit("⛔ داده‌ای برای آموزش پیدا نشد. ابتدا Cell 7 را اجرا کن و از وجود features.parquet مطمئن شو.")

X_train_df = pd.concat(train_parts, ignore_index=True).fillna(0)
print(f"Training rows (requested cap): {len(X_train_df):,} | Features: {len(feature_cols)}")

# 3) کاهش مصرف حافظه: تبدیل به float32 قبل از اسکِیل
X_train = X_train_df.values.astype(np.float32, copy=False)
# آزادکردن حافظهٔ DataFrame (جا برای آرایه‌ها و مدل)
del X_train_df, train_parts; gc.collect()

# 4) نرمال‌سازی با RobustScaler (خروجی معمولاً float64 است)
X_train_scaled = scaler.transform(X_train)
# برای کاهش RAM هنگام آموزش، می‌توانیم cast کنیم به float32
X_train_scaled = X_train_scaled.astype(np.float32, copy=False)

# 5) آموزش IsolationForest با توان بالاتر
# - n_estimators=500: دقت بهتر از 300، با هزینهٔ زمان بیشتر
# - max_samples=512k: هر درخت روی نمونهٔ بزرگ‌تر → پوشش بهتر رفتارهای نادر
# - n_jobs=-1: استفاده از همهٔ هسته‌های در دسترس
iforest = IsolationForest(
    n_estimators=500,
    max_samples=min(512_000, X_train_scaled.shape[0]),
    contamination='auto',
    max_features=1.0,
    bootstrap=False,
    n_jobs=-1,
    random_state=42,
    verbose=1
)
iforest.fit(X_train_scaled)

joblib.dump(iforest, MODEL_OUT)
print(f"✅ IsolationForest saved to: {MODEL_OUT}")


Training rows (requested cap): 2,600,000 | Features: 32


[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done   2 out of   2 | elapsed:  4.3min finished


✅ IsolationForest saved to: /content/data/iforest_model.joblib


In [ ]:
# Cell 9: Score all rows in chunks (clean)
# هدف: کل دیتاست فیچر را چانکی می‌خوانیم، با RobustScaler نرمال می‌کنیم و با IsolationForest نمره می‌دهیم.
# خروجی: /content/data/scores.parquet با ستون‌های row_id و raw_if (بزرگ‌تر = غیرعادی‌تر)

import os
import json
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.dataset as ds
import joblib

FEATURES_IN   = "/content/data/features.parquet"
SCALER_IN     = "/content/data/robust_scaler.joblib"
FEATURE_LIST  = "/content/data/feature_list.json"
MODEL_IN      = "/content/data/iforest_model.joblib"
SCORES_OUT    = "/content/data/scores.parquet"
BATCH_SIZE    = 200_000

# 1) حذف فایل خروجی قبلی در صورت وجود
if os.path.exists(SCORES_OUT):
    try:
        os.remove(SCORES_OUT)
    except Exception as e:
        print(f"⚠️ Warning: could not remove existing {SCORES_OUT}: {e}")

# 2) بارگذاری منابع
feat_ds = ds.dataset(FEATURES_IN, format="parquet")
scaler  = joblib.load(SCALER_IN)
iforest = joblib.load(MODEL_IN)
with open(FEATURE_LIST, "r") as f:
    feature_cols = json.load(f)["features"]

# 3) نوشتن چانکی به پارکت
writer = None
schema_scores = None
rows_scored = 0

for batch in feat_ds.to_batches(batch_size=BATCH_SIZE):
    dfb = batch.to_pandas()

    row_ids = dfb["row_id"].astype("int64")
    Xb = dfb[feature_cols].fillna(0).values
    Xb_scaled = scaler.transform(Xb)

    # sklearn: score_samples بالاتر = نرمال‌تر ⇒ منفی می‌کنیم تا بزرگ‌تر = غیرعادی‌تر
    raw_if = (-iforest.score_samples(Xb_scaled)).astype(np.float32)

    out_df = pd.DataFrame({
        "row_id": row_ids.values,
        "raw_if": raw_if
    })

    table = pa.Table.from_pandas(out_df, preserve_index=False)
    if writer is None:
        schema_scores = table.schema
        writer = pq.ParquetWriter(SCORES_OUT, schema_scores)
    else:
        if table.schema != schema_scores:
            table = table.cast(schema_scores)
    writer.write_table(table)

    rows_scored += len(out_df)
    print(f"✅ Scored: {rows_scored:,} rows total")

if writer is not None:
    writer.close()

print(f"\n✅ Scores written to: {SCORES_OUT}")

[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.8s
[Parallel(n_jobs=1)]: Done 199 tasks      | elapsed:    3.9s
[Parallel(n_jobs=1)]: Done 449 tasks      | elapsed:   11.4s
[Parallel(n_jobs=1)]: Done 500 out of 500 | elapsed:   13.6s finished


✅ Scored: 200,000 rows total


[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.8s
[Parallel(n_jobs=1)]: Done 199 tasks      | elapsed:    3.4s
[Parallel(n_jobs=1)]: Done 449 tasks      | elapsed:    8.8s
[Parallel(n_jobs=1)]: Done 500 out of 500 | elapsed:    9.7s finished


✅ Scored: 400,000 rows total


[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.9s
[Parallel(n_jobs=1)]: Done 199 tasks      | elapsed:    3.5s
[Parallel(n_jobs=1)]: Done 449 tasks      | elapsed:    8.3s
[Parallel(n_jobs=1)]: Done 500 out of 500 | elapsed:    9.5s finished


...

[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.8s
[Parallel(n_jobs=1)]: Done 199 tasks      | elapsed:    3.3s
[Parallel(n_jobs=1)]: Done 449 tasks      | elapsed:    7.5s
[Parallel(n_jobs=1)]: Done 500 out of 500 | elapsed:    8.5s finished


✅ Scored: 4,000,000 rows total


[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.3s
[Parallel(n_jobs=1)]: Done 199 tasks      | elapsed:    1.3s
[Parallel(n_jobs=1)]: Done 449 tasks      | elapsed:    2.9s


✅ Scored: 4,046,428 rows total

✅ Scores written to: /content/data/scores.parquet


[Parallel(n_jobs=1)]: Done 500 out of 500 | elapsed:    3.1s finished


In [ ]:
# Cell 10: Calibrate threshold (0.0003) + compute severity 0..99
# هدف: روی raw_if کل دیتاست آستانه تعیین کنیم (~0.03% بالا)، سپس severity 0..99 و برچسب anomaly را بسازیم.
# خروجی: /content/data/scores_calibrated.parquet

import os
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.dataset as ds

SCORES_IN = "/content/data/scores.parquet"
CALIBRATED_OUT = "/content/data/scores_calibrated.parquet"

# 1) بارگذاری کل نمره‌ها (۴ میلیون ردیف ~ 16MB)
score_ds = ds.dataset(SCORES_IN, format="parquet")
all_scores = []
all_row_ids = []
for b in score_ds.to_batches(batch_size=500_000):
    dfb = b.to_pandas()
    all_row_ids.append(dfb["row_id"].values)
    all_scores.append(dfb["raw_if"].values)

if not all_scores:
    raise SystemExit("⛔ هیچ نمره‌ای بارگذاری نشد. ابتدا Cell 9 را اجرا کن.")

row_ids = np.concatenate(all_row_ids)
raw_if  = np.concatenate(all_scores)
N = raw_if.shape[0]

# 2) آستانه برای ~0.03% بالایی (0.0003)
q = 0.9997
threshold = float(np.quantile(raw_if, q))

# 3) محاسبه percentile rank برای هر رکورد
order = np.argsort(raw_if)
ranks = np.empty_like(order)
ranks[order] = np.arange(N)
percentile = (ranks + 1) / N

# 4) نگاشت severity (0..99) با فشرده‌سازی دم بالا (لجیت)
a = 10.0
b = 0.9
sev = 1.0 / (1.0 + np.exp(-a * (percentile - b)))
sev = np.clip(sev, 0, 0.9999)
severity = np.floor(100.0 * sev).astype(np.int16)

# 5) تعیین برچسب anomaly
is_anomaly = (raw_if >= threshold)

calib_df = pd.DataFrame({
    "row_id": row_ids.astype(np.int64),
    "raw_if": raw_if.astype(np.float32),
    "percentile": percentile.astype(np.float32),
    "severity": severity.astype(np.int16),
    "is_anomaly": is_anomaly.astype(np.int8),
    "threshold": np.full(N, threshold, dtype=np.float32)
})

# 6) ذخیره خروجی
if os.path.exists(CALIBRATED_OUT):
    try:
        os.remove(CALIBRATED_OUT)
    except Exception as e:
        print(f"⚠️ Warning: could not remove {CALIBRATED_OUT}: {e}")

pq.write_table(pa.Table.from_pandas(calib_df, preserve_index=False), CALIBRATED_OUT)

print(f"✅ Calibrated scores written to: {CALIBRATED_OUT}")
print(f"🔎 Threshold (99.97% quantile): {threshold:.4f}")
print(f"📊 Total rows: {N:,}")

✅ Calibrated scores written to: /content/data/scores_calibrated.parquet
🔎 Threshold (99.97% quantile): 0.6762
📊 Total rows: 4,046,428


In [ ]:
# Cell 11: Package & download model bundle
import json, os, zipfile
import pyarrow.dataset as ds
from google.colab import files

# Paths
MODEL      = "/content/data/iforest_model.joblib"
SCALER     = "/content/data/robust_scaler.joblib"
FEATURESJS = "/content/data/feature_list.json"
SCORES_CAL = "/content/data/scores_calibrated.parquet"
BUNDLE_ZIP = "/content/model_bundle.zip"
META_JSON  = "/content/model_metadata.json"

# 1) خواندن threshold از scores_calibrated (از اولین batch)
th = None
score_ds = ds.dataset(SCORES_CAL, format="parquet")
for b in score_ds.to_batches(batch_size=1):
    dfb = b.to_pandas()
    th = float(dfb["threshold"].iloc[0])
    break
if th is None:
    raise SystemExit("⛔ نتوانستم threshold را از scores_calibrated بخوانم. ابتدا Cell 10 را اجرا کن.")

# 2) نوشتن متادیتای مدل
metadata = {
    "threshold": th,
    "severity_mapping": {"type": "logistic", "a": 10.0, "b": 0.9},
    "quantile_for_threshold": 0.9997,
    "notes": "raw_if = -iforest.score_samples(RobustScaler.transform(features))"
}
with open(META_JSON, "w") as f:
    json.dump(metadata, f, indent=2)

# 3) ساختن zip باندل
for p in [MODEL, SCALER, FEATURESJS, META_JSON]:
    if not os.path.exists(p):
        raise SystemExit(f"⛔ فایل لازم پیدا نشد: {p}")

if os.path.exists(BUNDLE_ZIP):
    os.remove(BUNDLE_ZIP)

with zipfile.ZipFile(BUNDLE_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(MODEL,      arcname=os.path.basename(MODEL))
    z.write(SCALER,     arcname=os.path.basename(SCALER))
    z.write(FEATURESJS, arcname=os.path.basename(FEATURESJS))
    z.write(META_JSON,  arcname=os.path.basename(META_JSON))

print(f"✅ Bundle ready: {BUNDLE_ZIP}")
files.download(BUNDLE_ZIP)  # دانلود در Colab


✅ Bundle ready: /content/model_bundle.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation

# اگر دیتات خیلی بزرگه، برای انیمیشن یه نمونه کوچیک‌تر استفاده کن (اختیاری)
plot_df_anim = plot_df.sample(n=120_000, random_state=42).reset_index(drop=True)

# محورها: PC2 - PC3
x = plot_df_anim["PC2c"].values
y = plot_df_anim["PC3c"].values
labels = plot_df_anim["is_anomaly"].values  # 0 = normal, 1 = anomaly

N = len(x)

# ✅ تعداد فریم‌ها: کمتر از تعداد نقطه، ولی همه‌ی نقاط تا آخر فریم‌ها نمایش داده می‌شن
N_FRAMES = 280  # اگر خیلی تند/کند بود، این عدد رو کم/زیاد کن

fig, ax = plt.subplots(figsize=(6, 6))

scat = ax.scatter([], [], s=4)

ax.set_xlim(x.min(), x.max())
ax.set_ylim(y.min(), y.max())
ax.set_xlabel("PC2 (clipped)")
ax.set_ylabel("PC3 (clipped)")
ax.set_title("Animated PCA — PC2 vs PC3")

def init():
    empty = np.empty((0, 2))
    scat.set_offsets(empty)
    return scat,

def update(frame):
    # درصد پیشرفت انیمیشن (بین 0 و 1)
    frac = (frame + 1) / N_FRAMES
    # تعداد نقاطی که تا این فریم باید دیده بشن
    k = max(1, min(N, int(frac * N)))

    pts = np.column_stack([x[:k], y[:k]])  # شکل (k, 2)
    scat.set_offsets(pts)

    # رنگ‌دهی: نرمال‌ها آبی، انومالی‌ها قرمز
    c = np.where(labels[:k] == 1, "red", "blue")
    scat.set_color(c)

    return scat,

anim = animation.FuncAnimation(
    fig,
    update,
    init_func=init,
    frames=N_FRAMES,
    interval=20,   # فاصله فریم‌ها (ms) — اگر خواستی سریع‌تر بشه تا 10 هم می‌تونی کمش کنی
    blit=True
)

plt.close(fig)

# ذخیره به صورت MP4
anim.save("pca_pc2_pc3_animation.mp4", writer="ffmpeg", fps=30)

from google.colab import files
files.download("pca_pc2_pc3_animation.mp4")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Cell 13: Export all anomalies (raw log columns + severity/raw_if/percentile) to Excel — no row_id in raw file required
# هدف: همه‌ی انومالی‌ها (~1200 رکورد) را با ستون‌های خام اصلیِ لاگ + ستون‌های مدل (severity/raw_if/percentile)
#       در یک فایل اکسل خروجی بگیریم و دانلود کنیم؛ حتی اگر cleaned_data.parquet ستون row_id نداشته باشد.

import os
import json
import numpy as np
import pandas as pd
import pyarrow.dataset as ds
from google.colab import files

# مسیرها
RAW_PARQUET   = "/content/data/cleaned_data.parquet"   # لاگ خام تمیزشده (Cell 5)
SCORES_CAL_IN = "/content/data/scores_calibrated.parquet"
XLSX_OUT      = "/content/anomalies_raw_export.xlsx"
BATCH_SIZE    = 200_000

# ستون‌های خام موردنیاز به همان ترتیبی که گفتی
RAW_COLS = [
    "@timestamp",
    "source.address",
    "destination.address",
    "destination.port",
    "suricata.eve.alert.signature",
    "message",
    "event.type",
    "http.response.status_code",
    "rule.category",
    "rule.name",
    "url.original",
    "user_agent.original",
    "suricata.eve.alert.metadata",
    "source_file",
]

# 1) خواندن row_id انومالی‌ها + ستون‌های نمره (از scores_calibrated)
score_ds = ds.dataset(SCORES_CAL_IN, format="parquet")
rows = []
for b in score_ds.to_batches(batch_size=BATCH_SIZE):
    dfb = b.to_pandas()
    sel = dfb[dfb["is_anomaly"] == 1][["row_id", "raw_if", "severity", "percentile"]]
    if not sel.empty:
        rows.append(sel)

if not rows:
    raise SystemExit("⛔ هیچ انومالی پیدا نشد. ابتدا Cell 10 را اجرا کن.")

anom_scores = pd.concat(rows, ignore_index=True)
anom_scores["row_id"] = anom_scores["row_id"].astype(np.int64)
row_ids_set = set(anom_scores["row_id"].tolist())
print(f"Found anomalies: {len(anom_scores):,}")

# 2) پیمایش cleaned_data.parquet و ساخت row_id جاری بر اساس offset
raw_ds = ds.dataset(RAW_PARQUET, format="parquet")
parts = []
offset = 0

for b in raw_ds.to_batches(batch_size=BATCH_SIZE):
    dfb = b.to_pandas()
    n = len(dfb)
    # row_id جریانـی مطابق همان ترتیبی که در Cell 6 استفاده شده بود
    rid = np.arange(offset, offset + n, dtype=np.int64)
    offset += n

    # فقط ستون‌های خام موجود را نگه می‌داریم
    have = [c for c in RAW_COLS if c in dfb.columns]
    dfb = dfb[have]

    # انتخاب رکوردهایی که row_idشان در مجموعه‌ی انومالی‌هاست
    mask = np.isin(rid, list(row_ids_set))
    if mask.any():
        picked = dfb.loc[mask].copy()
        picked.insert(0, "row_id", rid[mask])
        parts.append(picked)

if not parts:
    raise SystemExit("⛔ نتوانستم رکوردهای خام مربوط به انومالی‌ها را در cleaned_data.parquet پیدا کنم (بر اساس row_id جریانـی).")

raw_sel = pd.concat(parts, ignore_index=True)
print(f"Matched raw rows: {len(raw_sel):,}")

# 3) Merge نهایی روی row_id و مرتب‌سازی بر اساس severity/raw_if
out_df = raw_sel.merge(anom_scores, on="row_id", how="inner")
out_df = out_df.sort_values(["severity", "raw_if"], ascending=[False, False])

# 4) ترتیب ستون‌ها: اول خام‌ها، بعد ستون‌های مدل
ordered_cols = [
    "@timestamp",
    "source.address",
    "destination.address",
    "destination.port",
    "suricata.eve.alert.signature",
    "message",
    "event.type",
    "http.response.status_code",
    "rule.category",
    "rule.name",
    "url.original",
    "user_agent.original",
    "suricata.eve.alert.metadata",
    "source_file",
    "severity",
    "raw_if",
    "percentile",
]
ordered_cols = [c for c in ordered_cols if c in out_df.columns]

# 5) نوشتن به اکسل و دانلود
if os.path.exists(XLSX_OUT):
    try:
        os.remove(XLSX_OUT)
    except Exception as e:
        print(f"⚠️ Warning: could not remove existing {XLSX_OUT}: {e}")

with pd.ExcelWriter(XLSX_OUT, engine="openpyxl") as writer:
    out_df[ordered_cols].to_excel(writer, sheet_name="anomalies", index=False)

print(f"✅ Excel written: {XLSX_OUT}")
files.download(XLSX_OUT)  # شروع دانلود در Colab


Found anomalies: 1,216
Matched raw rows: 1,216
✅ Excel written: /content/anomalies_raw_export.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Cell A: Upload ZIP, extract fresh to /content/upload, pick largest CSV, preview columns

import os, zipfile, shutil
import pandas as pd
from google.colab import files

# ستون‌های خام موردنیاز؛ اگر در فایل نباشند بعداً ستون خالی ساخته می‌شود
RAW_COLS = [
    '@timestamp','source.address','destination.address','destination.port',
    'suricata.eve.alert.signature','message','event.type','http.response.status_code',
    'rule.category','rule.name','url.original','user_agent.original',
    'suricata.eve.alert.metadata','source_file'
]

# 1) آپلود ZIP
print("📦 Please select your ZIP (containing CSV) to upload…")
up = files.upload()
ZIP_NAME = next((k for k in up.keys() if k.lower().endswith('.zip')), None)
if not ZIP_NAME:
    raise SystemExit("⛔ No .zip uploaded. Please upload a ZIP containing your CSV.")

# 2) حذف کامل پوشه‌ی قبلی و اکسترکت جدید
EXTRACT_DIR = "/content/upload"
if os.path.exists(EXTRACT_DIR):
    shutil.rmtree(EXTRACT_DIR)   # حذف کل پوشه و محتویاتش
os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_NAME, 'r') as z:
    z.extractall(EXTRACT_DIR)

# 3) فهرست CSVها و انتخاب بزرگ‌ترین
csv_paths = []
for root, _, files_ in os.walk(EXTRACT_DIR):
    for fn in files_:
        if fn.lower().endswith(".csv"):
            p = os.path.join(root, fn)
            try:
                sz = os.path.getsize(p)
            except:
                sz = -1
            csv_paths.append((p, sz))

if not csv_paths:
    raise SystemExit("⛔ No CSV found inside the ZIP.")

csv_paths.sort(key=lambda x: x[1], reverse=True)
CSV_PATH = csv_paths[0][0]
print(f"✅ Selected CSV: {CSV_PATH} (largest in ZIP)")

# 4) Preflight peek
peek = pd.read_csv(CSV_PATH, nrows=5, dtype=str)
used_cols = [c for c in RAW_COLS if c in peek.columns]
missing_cols = [c for c in RAW_COLS if c not in peek.columns]

print("🧭 Columns in file:", list(peek.columns))
print("✅ Will use these columns:", used_cols)
if missing_cols:
    print("⚠️ Missing columns (will be created as empty):", missing_cols)


📦 Please select your ZIP (containing CSV) to upload…


Saving Untitled Discover session (7).zip to Untitled Discover session (7).zip
✅ Selected CSV: /content/upload/Untitled Discover session (7).csv (largest in ZIP)
🧭 Columns in file: ['@timestamp', '_id', '_ignored', '_index', '_score', 'agent.ephemeral_id', 'agent.id', 'agent.name', 'agent.name.text', 'agent.type', 'agent.version', 'data_stream.dataset', 'data_stream.namespace', 'data_stream.type', 'destination.address', 'destination.as.number', 'destination.as.organization.name', 'destination.as.organization.name.text', 'destination.bytes', 'destination.domain', 'destination.geo.city_name', 'destination.geo.continent_name', 'destination.geo.country_iso_code', 'destination.geo.country_name', 'destination.geo.location', 'destination.geo.region_iso_code', 'destination.geo.region_name', 'destination.ip', 'destination.packets', 'destination.port', 'ecs.version', 'elastic_agent.id', 'elastic_agent.snapshot', 'elastic_agent.version', 'event.agent_id_status', 'event.category', 'event.created', 

In [ ]:
import json, os

META_JSON = "/content/data/model_metadata.json"
os.makedirs("/content/data", exist_ok=True)

metadata = {
    "threshold": 0.6762,
    "severity_mapping": {"type": "logistic", "a": 10.0, "b": 0.9},
    "quantile_for_threshold": 0.9997,
    "notes": "raw_if = -iforest.score_samples(RobustScaler.transform(features))"
}

with open(META_JSON, "w") as f:
    json.dump(metadata, f, indent=2)

print(f"✅ model_metadata.json recreated at {META_JSON}")


✅ model_metadata.json recreated at /content/data/model_metadata.json


In [ ]:
# 1) مسیر دیتای داخل Drive
drive_data = "/content/drive/MyDrive/colab_outputs/data"

# 2) مطمئن شو وجود داره
import os
assert os.path.exists(drive_data), f"Drive data folder not found: {drive_data}"

# 3) هرچی قبلاً بوده پاک و لینک بساز
if os.path.islink("/content/data") or os.path.exists("/content/data"):
    # حذف پوشه/لینک قدیمی
    import shutil
    try:
        os.remove("/content/data")
    except IsADirectoryError:
        shutil.rmtree("/content/data")

os.symlink(drive_data, "/content/data")

# 4) تست سریع: فایل‌های کلیدی رو چک کن
for fn in ["robust_scaler.joblib","iforest_model.joblib","feature_list.json","model_metadata.json"]:
    p = f"/content/data/{fn}"
    print(("✔" if os.path.exists(p) else "✘"), p)


✔ /content/data/robust_scaler.joblib
✔ /content/data/iforest_model.joblib
✔ /content/data/feature_list.json
✔ /content/data/model_metadata.json


In [ ]:
# Cell B — No-line-drop robust scoring (custom CSV reader, two-pass)
import os, re, math, json, gc, io, csv
import numpy as np
import pandas as pd
import joblib
from urllib.parse import urlparse
from google.colab import files

# ===== Mount/link: make /content/data point to Drive if present (بدون تغییر در منطق) =====
DRIVE_DATA = "/content/drive/MyDrive/colab_outputs/data"
if os.path.exists(DRIVE_DATA):
    if os.path.islink("/content/data") or os.path.exists("/content/data"):
        import shutil
        try:
            os.remove("/content/data")
        except IsADirectoryError:
            shutil.rmtree("/content/data")
    os.symlink(DRIVE_DATA, "/content/data")

# ===== Paths (use existing vars if defined) =====
SCALER_IN   = globals().get("SCALER_IN", "/content/data/robust_scaler.joblib")
MODEL_IN    = globals().get("MODEL_IN", "/content/data/iforest_model.joblib")
FEATURE_JS  = globals().get("FEATURE_JS", "/content/data/feature_list.json")
META_JSON   = globals().get("META_JSON", "/content/data/model_metadata.json")

# خروجی پیش‌فرض روی Drive (اگر قبلاً OUT_CSV ست نشده)
DEFAULT_OUT = "/content/drive/MyDrive/colab_outputs/inference_scored_with_severity.csv"
OUT_CSV     = globals().get("OUT_CSV", DEFAULT_OUT)

# ===== Preconditions =====
for p in [SCALER_IN, MODEL_IN, FEATURE_JS, META_JSON]:
    if not os.path.exists(p):
        raise SystemExit(f"⛔ File not found: {p}")
try:
    CSV_PATH
except NameError:
    raise SystemExit("⛔ CSV_PATH is not set. Run Cell A first.")

# ===== Helpers for feature list (حداقل تغییر: پذیرش list یا dict) =====
def _read_feature_list(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        obj = json.load(f)
    if isinstance(obj, list):
        return [c for c in obj if isinstance(c, str) and c.strip()]
    if isinstance(obj, dict):
        for k in ("features","feature_names","columns","selected_features"):
            v = obj.get(k)
            if isinstance(v, list):
                return [c for c in v if isinstance(c, str) and c.strip()]
    return []

def _cols_from_parquet(p):
    if os.path.exists(p):
        try:
            df = pd.read_parquet(p)
            return [c for c in df.columns if isinstance(c, str) and c.strip()]
        except Exception:
            pass
    return []

# ===== Load artifacts =====
scaler  = joblib.load(SCALER_IN)
iforest = joblib.load(MODEL_IN)

# حداقل تغییر: منبع فیچرها -> JSON سپس پارکت‌ها
FEATURE_COLS = _read_feature_list(FEATURE_JS)
if not FEATURE_COLS:
    FEATURE_COLS = _cols_from_parquet("/content/data/features.parquet")
if not FEATURE_COLS:
    FEATURE_COLS = _cols_from_parquet("/content/data/cleaned_data.parquet")
if not FEATURE_COLS:
    raise SystemExit("⛔ Could not resolve feature list from JSON/parquet files.")

with open(META_JSON, "r", encoding="utf-8") as f:
    META = json.load(f)

# هم‌راستاسازی فیچرها با مدل/اسکیلر (همان منطق با حداقل تغییر)
model_names  = getattr(iforest, "feature_names_in_", None)
scaler_names = getattr(scaler,  "feature_names_in_", None)
model_n      = getattr(iforest, "n_features_in_", None)

if model_names is not None and len(model_names) > 0:
    s = set(FEATURE_COLS)
    FEATURE_COLS = [n for n in model_names if n in s]

if model_n is not None and len(FEATURE_COLS) != model_n and scaler_names is not None:
    s = set(FEATURE_COLS)
    aligned = [n for n in scaler_names if n in s]
    if aligned:
        FEATURE_COLS = aligned

if model_n is not None and len(FEATURE_COLS) != model_n and os.path.exists("/content/data/features.parquet"):
    cols = [c for c in pd.read_parquet("/content/data/features.parquet").columns if isinstance(c, str)]
    s = set(FEATURE_COLS)
    inter = [c for c in cols if c in s]
    if inter:
        FEATURE_COLS = inter[:model_n] if model_n else inter

if model_n is not None:
    assert len(FEATURE_COLS) == model_n, f"⛔ Feature count mismatch: have {len(FEATURE_COLS)}, model expects {model_n}"

THRESHOLD = float(META.get("threshold", 0.0))
A = float(META.get("severity_mapping", {}).get("a", 10.0))
B = float(META.get("severity_mapping", {}).get("b", 0.9))

# ===== Raw columns we need from the CSV =====
RAW_COLS = [
    '@timestamp','source.address','destination.address','destination.port',
    'suricata.eve.alert.signature','message','event.type','http.response.status_code',
    'rule.category','rule.name','url.original','user_agent.original',
    'suricata.eve.alert.metadata','source_file'
]

# ===== Helpers (حداقل تغییر: safe_len و ua_flags مقاوم به pd.NA) =====
def is_private_ip(ip: str) -> int:
    if not isinstance(ip, str): return 0
    if ip.startswith("10.") or ip.startswith("192.168."): return 1
    if ip.startswith("172."):
        try:
            sec = int(ip.split(".")[1]); return 1 if 16 <= sec <= 31 else 0
        except: return 0
    return 0

def ip_to_24(ip: str) -> float:
    try:
        o = ip.split("."); return float(int(o[0])*256*256 + int(o[1])*256 + int(o[2]))
    except: return np.nan

def safe_len(x) -> int:
    # مقاوم در برابر pd.NA/NaN/None
    try:
        import pandas as _pd
        if x is None or (_pd.isna(x)):
            return 0
    except Exception:
        if x is None:
            return 0
    try:
        return len(str(x))
    except Exception:
        return 0

def char_entropy(s: str) -> float:
    if not isinstance(s, str) or not s: return 0.0
    from collections import Counter
    c = Counter(s); n = sum(c.values())
    ps = [v/n for v in c.values()]
    return float(-sum(p*math.log2(p) for p in ps))

def count_chars(s: str, chars='/?&=') -> int:
    if not isinstance(s, str): return 0
    return sum(s.count(ch) for ch in chars)

def url_is_host_ip(u: str) -> int:
    try:
        host = urlparse(u).hostname
        if not host: return 0
        return 1 if re.fullmatch(r"\d{1,3}(\.\d{1,3}){3}", host) else 0
    except: return 0

def ua_flags(ua) -> pd.Series:
    # جایگزین (ua or "") که با pd.NA مشکل داشت
    try:
        import pandas as _pd
        if ua is None or _pd.isna(ua):
            ua_str = ""
        else:
            ua_str = str(ua)
    except Exception:
        ua_str = "" if ua is None else str(ua)
    ua_low = ua_str.lower()
    return pd.Series({
        "ua_len": safe_len(ua_low),
        "ua_is_browser": int(any(k in ua_low for k in ["chrome","safari","firefox","edge"])),
        "ua_is_lib": int(any(k in ua_low for k in ["curl","python-requests","wget","go-http-client"])),
        "ua_is_mobile": int(any(k in ua_low for k in ["android","iphone","ipad"]))
    })

# زمان: حداقل تغییر (حذف view و استفاده از astype)
def _parse_ts(series: pd.Series) -> pd.Series:
    ts = pd.to_datetime(series, errors="coerce", format="%b %d, %Y @ %H:%M:%S.%f")
    bad = ts.isna()
    if bad.any():
        ts2 = pd.to_datetime(series[bad], errors="coerce")
        ts[bad] = ts2
    return ts

def build_features(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)
    # time
    ts = _parse_ts(df["@timestamp"])
    ns = ts.astype('int64', copy=False).astype('float64')  # NaT → -9e18 → float
    ns[ts.isna()] = np.nan
    out["epoch_ms"] = (ns / 1_000_000.0).astype("float64")
    out["hour"] = ts.dt.hour.astype("Int16")
    out["dow"]  = ts.dt.dayofweek.astype("Int16")
    # IPs
    out["src_priv"] = df["source.address"].apply(is_private_ip).astype("Int8")
    out["dst_priv"] = df["destination.address"].apply(is_private_ip).astype("Int8")
    out["src_24"]   = df["source.address"].apply(ip_to_24).astype("float32")
    out["dst_24"]   = df["destination.address"].apply(ip_to_24).astype("float32")
    out["src_eq_dst"] = (df["source.address"] == df["destination.address"]).astype("Int8")
    # Ports / HTTP
    ports_clean = df["destination.port"].astype(str).str.replace(",","", regex=False)
    out["dst_port"] = pd.to_numeric(ports_clean, errors="coerce").astype("float64")
    out["port_wellknown"] = (out["dst_port"] <= 1024).fillna(False).astype("Int8")
    out["port_ephemeral"] = (out["dst_port"] >= 49152).fillna(False).astype("Int8")
    out["http_status"] = pd.to_numeric(df["http.response.status_code"], errors="coerce").astype("float64")
    # URL (حداقل تغییر: fillna → str)
    url = df["url.original"]
    try: url = url.fillna("")
    except Exception: pass
    url = url.astype(str)
    out["url_len"]      = url.apply(safe_len).astype("Int32")
    out["url_marks"]    = url.apply(count_chars).astype("Int32")
    out["url_entropy"]  = url.apply(char_entropy).astype("float32")
    out["url_host_is_ip"] = url.apply(url_is_host_ip).astype("Int8")
    # UA (حداقل تغییر: fillna → apply(ua_flags))
    ua = df["user_agent.original"]
    try: ua = ua.fillna("")
    except Exception: pass
    ua_feat = ua.apply(ua_flags)
    out = pd.concat([out, ua_feat], axis=1)
    out["ua_len"]        = out["ua_len"].astype("Int32")
    out["ua_is_browser"] = out["ua_is_browser"].astype("Int8")
    out["ua_is_lib"]     = out["ua_is_lib"].astype("Int8")
    out["ua_is_mobile"]  = out["ua_is_mobile"].astype("Int8")
    # Suricata text lengths + missing (حداقل تغییر: fillna(""))
    for col in [
        "suricata.eve.alert.signature","rule.name","rule.category",
        "suricata.eve.alert.metadata","message","event.type"
    ]:
        s = df[col]
        try: s = s.fillna("")
        except Exception: pass
        s = s.astype(str)
        out[col.replace(".","_")+"_len"]     = s.apply(safe_len).astype("Int32")
        out[col.replace(".","_")+"_missing"] = (s == "").astype("Int8")
    return out

# ===== Custom CSV Reader (no row is dropped) =====
def detect_delimiter_from_sample(path, max_bytes=2_000_000):
    with open(path, 'rb') as f:
        sample = f.read(max_bytes)
    txt = sample.decode('utf-8', errors='ignore')
    try:
        dialect = csv.Sniffer().sniff(txt, delimiters=[',',';','\t','|'])
        return dialect.delimiter
    except Exception:
        for line in txt.splitlines():
            if line.strip():
                counts = {d: line.count(d) for d in [',',';','\t','|']}
                return max(counts, key=counts.get)
        return ','

def row_adjust_to_header(row, header_len, delimiter):
    if len(row) > header_len:
        keep = row[:header_len-1]
        last = delimiter.join(row[header_len-1:])
        return keep + [last]
    elif len(row) < header_len:
        return row + ['']*(header_len - len(row))
    return row

def stream_csv_as_chunks(path, chunk_rows=200_000, encoding='utf-8'):
    delim = detect_delimiter_from_sample(path)
    print(f"ℹ️ Using delimiter: {repr(delim)}")
    with open(path, 'r', encoding=encoding, errors='ignore', newline='') as f:
        reader = csv.reader(f, delimiter=delim, quotechar='"', doublequote=True, escapechar='\\')
        # header
        for hdr in reader:
            if not hdr or all((h is None or str(h).strip() == '') for h in hdr):
                continue
            header = [str(h) for h in hdr]
            break
        else:
            raise SystemExit("⛔ Could not read header row.")
        header_len = len(header)
        acc = []
        for row in reader:
            row = row_adjust_to_header(row, header_len, delim)
            acc.append(row)
            if len(acc) >= chunk_rows:
                df = pd.DataFrame(acc, columns=header, dtype=str)
                yield df
                acc.clear()
        if acc:
            df = pd.DataFrame(acc, columns=header, dtype=str)
            yield df

# ===== Pass 1: compute raw_if array over entire file =====
rawif_parts = []
n_total = 0

for i, raw in enumerate(stream_csv_as_chunks(CSV_PATH, chunk_rows=200_000), 1):
    for c in RAW_COLS:
        if c not in raw.columns:
            raw[c] = np.nan
    raw = raw[RAW_COLS]

    feats = build_features(raw)
    X = feats.reindex(columns=FEATURE_COLS, fill_value=np.nan).fillna(0).values
    X_scaled = scaler.transform(X)
    raw_if = (-iforest.score_samples(X_scaled)).astype(np.float32)

    rawif_parts.append(raw_if)
    n_total += len(raw_if)
    if i % 5 == 0:
        print(f"Pass1 — scored rows: {n_total:,}")
    del raw, feats, X, X_scaled, raw_if
    gc.collect()

if n_total == 0:
    raise SystemExit("⛔ No rows read from CSV. Check CSV_PATH/encoding/delimiter.")

raw_if_full = np.concatenate(rawif_parts, dtype=np.float32)
del rawif_parts
gc.collect()

# Percentile & severity across entire file
order = np.argsort(raw_if_full)
ranks = np.empty_like(order)
ranks[order] = np.arange(n_total, dtype=np.int64)
percentile_full = (ranks + 1) / float(n_total)

sev = 1.0 / (1.0 + np.exp(-A * (percentile_full - B)))
sev = np.clip(sev, 0, 0.9999)
severity_full = np.floor(100.0 * sev).astype(np.int16)

print(f"✅ Pass1 done. Total rows: {n_total:,}")

# ===== Pass 2: write full output (no rows dropped) =====
try:
    if os.path.exists(OUT_CSV):
        os.remove(OUT_CSV)
except Exception as e:
    print(f"⚠️ Could not remove old {OUT_CSV}: {e}")

start = 0
header_written = False
rows_written = 0

for i, raw in enumerate(stream_csv_as_chunks(CSV_PATH, chunk_rows=200_000), 1):
    for c in RAW_COLS:
        if c not in raw.columns:
            raw[c] = np.nan
    raw = raw[RAW_COLS]

    k = len(raw)
    end = start + k

    sev_chunk = severity_full[start:end]
    rif_chunk = raw_if_full[start:end]
    pct_chunk = percentile_full[start:end].astype(np.float32)

    out_df = raw.copy()
    out_df["severity"]   = sev_chunk
    out_df["raw_if"]     = rif_chunk
    out_df["percentile"] = pct_chunk

    out_df.to_csv(OUT_CSV, index=False, mode=("w" if not header_written else "a"),
                  header=(not header_written))
    header_written = True
    rows_written += k
    start = end

    if i % 5 == 0:
        print(f"Pass2 — written rows: {rows_written:,}")

del severity_full, raw_if_full, percentile_full
gc.collect()

print(f"✅ Saved: {OUT_CSV} (rows: {rows_written:,})")
try:
    files.download(OUT_CSV)
except Exception as e:
    print(f"ℹ️ Download skipped/failed (but file is saved on Drive): {e}")


ℹ️ Using delimiter: ','


[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done 199 tasks      | elapsed:    0.1s
[Parallel(n_jobs=1)]: Done 449 tasks      | elapsed:    0.2s
[Parallel(n_jobs=1)]: Done 500 out of 500 | elapsed:    0.2s finished


✅ Pass1 done. Total rows: 2,512
ℹ️ Using delimiter: ','
✅ Saved: /content/drive/MyDrive/colab_outputs/inference_scored_with_severity.csv (rows: 2,512)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Cell C (resilient & fast): Export anomalies + near-anomalies from scored CSV
# - Looks in Drive first; falls back to /content; else asks for upload
# - Requires columns: raw_if (必須), percentile (بهتر است؛ اگر نبود خطا می‌دهد)

import os, json
import numpy as np
import pandas as pd
from google.colab import files

# 1) Locate FULL_CSV robustly
CANDIDATES = [
    "/content/drive/MyDrive/colab_outputs/inference_scored_with_severity.csv",
    "/content/inference_scored_with_severity.csv",
]
FULL_CSV = None
for p in CANDIDATES:
    if os.path.exists(p):
        FULL_CSV = p
        break

if FULL_CSV is None:
    print("📥 Please upload your 'inference_scored_with_severity.csv' ...")
    up = files.upload()
    FULL_CSV = next(iter(up.keys()))
print(f"✅ Using file: {FULL_CSV}")

# 2) Load metadata (threshold)
META_JSON = "/content/data/model_metadata.json"
if os.path.exists(META_JSON):
    with open(META_JSON, "r", encoding="utf-8") as f:
        THRESHOLD = float(json.load(f).get("threshold", 0.6762))
else:
    THRESHOLD = 0.6762

# 3) Load CSV (must include raw_if; should include percentile)
df = pd.read_csv(FULL_CSV)
if "raw_if" not in df.columns:
    raise SystemExit("⛔ 'raw_if' در CSV نیست. Cell B را طوری اجرا کن که raw_if را ذخیره کند.")
if "percentile" not in df.columns:
    raise SystemExit("⛔ 'percentile' در CSV نیست. Cell B را طوری اجرا کن که percentile را ذخیره کند (برای سرعت و دقت).")

raw_if = df["raw_if"].to_numpy()
percentile = df["percentile"].to_numpy()

# 4) Masks: anomalies + near-anomalies (thin band below threshold)
mask_anom = (raw_if >= THRESHOLD)
q_thresh_file = float((raw_if < THRESHOLD).mean())  # ~0.9997 در حالت عادی
NEAR_WIDTH = 0.0002
lower_band = max(0.0, q_thresh_file - NEAR_WIDTH)
mask_near = (percentile >= lower_band) & (percentile < q_thresh_file)
mask_keep = mask_anom | mask_near

# 5) Ensure is_anomaly exists
if "is_anomaly" not in df.columns:
    df["is_anomaly"] = mask_anom.astype(np.int8)

# 6) Export
export_df = df.loc[mask_keep].copy().sort_values(
    by=["is_anomaly", "raw_if"], ascending=[False, False]
)
OUT_NEAR = "/content/inference_anomalies_plus_near.csv"
export_df.to_csv(OUT_NEAR, index=False)

print(f"Anomalies: {int(mask_anom.sum()):,} | Near anomalies: {int(mask_near.sum()):,}")
print(f"q_thresh_file={q_thresh_file:.6f}  band=[{lower_band:.6f}, {q_thresh_file:.6f})  width={NEAR_WIDTH:.6f}")
print(f"✅ Saved: {OUT_NEAR}")
try:
    files.download(OUT_NEAR)
except Exception as e:
    print(f"ℹ️ Download skipped/failed: {e}")


✅ Using file: /content/drive/MyDrive/colab_outputs/inference_scored_with_severity.csv
Anomalies: 5 | Near anomalies: 0
q_thresh_file=0.998010  band=[0.997810, 0.998010)  width=0.000200
✅ Saved: /content/inference_anomalies_plus_near.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
SAVE_DIR = "/content/drive/MyDrive/colab_outputs"
import os
os.makedirs(SAVE_DIR, exist_ok=True)


In [ ]:
!cp /content/inference_scored_with_severity.csv $SAVE_DIR/
!cp -r /content/data $SAVE_DIR/   # اگر پوشه داری


In [ ]:
import os

SAVE_DIR = "/content/drive/MyDrive/colab_outputs"  # پوشه‌ای که انتخاب کردی
for root, dirs, files in os.walk(SAVE_DIR):
    for f in files:
        print(os.path.join(root, f))


/content/drive/MyDrive/colab_outputs/inference_scored_with_severity.csv
/content/drive/MyDrive/colab_outputs/data/merged_all_parts.csv
/content/drive/MyDrive/colab_outputs/data/cleaned_data.parquet
/content/drive/MyDrive/colab_outputs/data/features.parquet
/content/drive/MyDrive/colab_outputs/data/robust_scaler.joblib
/content/drive/MyDrive/colab_outputs/data/feature_list.json
/content/drive/MyDrive/colab_outputs/data/iforest_model.joblib
/content/drive/MyDrive/colab_outputs/data/scores.parquet
/content/drive/MyDrive/colab_outputs/data/scores_calibrated.parquet
/content/drive/MyDrive/colab_outputs/data/model_metadata.json


In [ ]:
!du -h $SAVE_DIR


1.8G	/content/drive/MyDrive/colab_outputs/data
1.8G	/content/drive/MyDrive/colab_outputs


In [ ]:
!du -h $SAVE_DIR


1.8G	/content/drive/MyDrive/colab_outputs/data
1.8G	/content/drive/MyDrive/colab_outputs


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, joblib, hashlib

SAVE_DIR = "/content/drive/MyDrive/colab_outputs"
DATA_DIR = f"{SAVE_DIR}/data"

critical = {
    "SCALER_IN":   f"{DATA_DIR}/robust_scaler.joblib",
    "MODEL_IN":    f"{DATA_DIR}/iforest_model.joblib",
    "FEATURE_JS":  f"{DATA_DIR}/feature_list.json",
    "META_JSON":   f"{DATA_DIR}/model_metadata.json",
}

optional = [
    f"{SAVE_DIR}/inference_scored_with_severity.csv",
    f"{DATA_DIR}/merged_all_parts.csv",
    f"{DATA_DIR}/cleaned_data.parquet",
    f"{DATA_DIR}/features.parquet",
    f"{DATA_DIR}/scores.parquet",
    f"{DATA_DIR}/scores_calibrated.parquet",
]

def md5sum(p):
    h = hashlib.md5()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

print("=== Critical files ===")
for name, path in critical.items():
    assert os.path.exists(path), f"⛔ NOT FOUND: {name} at {path}"
    size = os.path.getsize(path)
    assert size > 0, f"⛔ EMPTY FILE: {name} at {path}"
    print(f"✔ {name}: {path}  ({size:,} bytes)  md5={md5sum(path)[:10]}")

print("\n=== Optional files (good to have) ===")
for path in optional:
    if os.path.exists(path):
        print(f"• Found: {path}  ({os.path.getsize(path):,} bytes)")
    else:
        print(f"• Missing (OK): {path}")

# Load artifacts
print("\n=== Loading artifacts ===")
scaler = joblib.load(critical["SCALER_IN"])
model  = joblib.load(critical["MODEL_IN"])
with open(critical["FEATURE_JS"], "r", encoding="utf-8") as f:
    feature_list = json.load(f)

assert isinstance(feature_list, (list, tuple)) and len(feature_list) > 0, "⛔ feature_list.json invalid"
print(f"✔ Scaler loaded")
print(f"✔ Model loaded: {type(model).__name__}")
print(f"✔ Features loaded: {len(feature_list)} features")

# Final output path on Drive
OUT_CSV = f"{SAVE_DIR}/inference_scored_with_severity.csv"
print(f"\n✅ OUT_CSV will be saved to: {OUT_CSV}")
print("\nALL GOOD ✅  — می‌تونی Colab رو ببندی. فردا با بوت‌استرپ زیر برگرد.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
=== Critical files ===
✔ SCALER_IN: /content/drive/MyDrive/colab_outputs/data/robust_scaler.joblib  (1,015 bytes)  md5=89f7325ded
✔ MODEL_IN: /content/drive/MyDrive/colab_outputs/data/iforest_model.joblib  (1,065,758,925 bytes)  md5=73504e2a57
✔ FEATURE_JS: /content/drive/MyDrive/colab_outputs/data/feature_list.json  (635 bytes)  md5=26aedf6c19
✔ META_JSON: /content/drive/MyDrive/colab_outputs/data/model_metadata.json  (222 bytes)  md5=1264b69aa9

=== Optional files (good to have) ===
• Found: /content/drive/MyDrive/colab_outputs/inference_scored_with_severity.csv  (1,183,395 bytes)
• Found: /content/drive/MyDrive/colab_outputs/data/merged_all_parts.csv  (612,668,186 bytes)
• Found: /content/drive/MyDrive/colab_outputs/data/cleaned_data.parquet  (34,330,466 bytes)
• Found: /content/drive/MyDrive/colab_outputs/data/features.parquet  (56,900,535 bytes)
• Found:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, joblib
import pandas as pd

SAVE_DIR = "/content/drive/MyDrive/colab_outputs"
DATA_DIR = f"{SAVE_DIR}/data"

FEATURE_JS  = f"{DATA_DIR}/feature_list.json"
FEATS_PQ    = f"{DATA_DIR}/features.parquet"
CLEANED_PQ  = f"{DATA_DIR}/cleaned_data.parquet"  # fallback
SCALER_IN   = f"{DATA_DIR}/robust_scaler.joblib"
MODEL_IN    = f"{DATA_DIR}/iforest_model.joblib"

def load_any_feature_list(path):
    with open(path, "r", encoding="utf-8") as f:
        obj = json.load(f)
    # حالت ۱: خودش لیست باشه
    if isinstance(obj, list):
        return [c for c in obj if isinstance(c, str) and c.strip()]
    # حالت ۲: دیکشنری با کلیدهای متداول
    if isinstance(obj, dict):
        for key in ["features", "feature_names", "columns", "selected_features"]:
            val = obj.get(key)
            if isinstance(val, list):
                return [c for c in val if isinstance(c, str) and c.strip()]
    return []

def cols_from_parquet(path):
    if os.path.exists(path):
        try:
            df = pd.read_parquet(path)
            return [c for c in df.columns if isinstance(c, str) and c.strip()]
        except Exception:
            pass
    return []

# 1) تلاش برای خواندن لیست فیچر از JSON
feat_list = load_any_feature_list(FEATURE_JS)

# 2) اگر خالی بود، از features.parquet یا cleaned_data.parquet بگیر
if not feat_list:
    feat_list = cols_from_parquet(FEATS_PQ)
if not feat_list:
    feat_list = cols_from_parquet(CLEANED_PQ)

assert len(feat_list) > 0, "⛔ نتونستم لیست فیچر رو از JSON یا پارکت‌ها استخراج کنم."

# 3) هم‌ترازی با مدل و اسکیلر
model  = joblib.load(MODEL_IN)
scaler = joblib.load(SCALER_IN)

def n_features(obj):
    return getattr(obj, "n_features_in_", None)

model_n  = n_features(model)
scaler_n = n_features(scaler)

# اگر مدل feature_names_in_ داشته باشه، بر اساس اون مرتب کنیم (ترجیح با مدل)
model_names = getattr(model, "feature_names_in_", None)
if model_names is not None and len(model_names) > 0:
    # فقط اشتراک رو نگه داریم و به همان ترتیب مدل
    set_feats = set(feat_list)
    aligned = [name for name in model_names if name in set_feats]
    if len(aligned) > 0:
        feat_list = aligned

# اگر هنوز طول با n_features_in_ برابر نیست، سعی کن با اسکیلر هم‌راستا کنی
scaler_names = getattr(scaler, "feature_names_in_", None)
if (model_n is not None and len(feat_list) != model_n) and scaler_names is not None:
    set_feats = set(feat_list)
    aligned = [name for name in scaler_names if name in set_feats]
    if len(aligned) > 0:
        feat_list = aligned

# گزارش نهایی و چک‌ها
msg = []
msg.append(f"Features resolved: {len(feat_list)}")
if model_n is not None:
    msg.append(f"Model expects:    {model_n}  ({'OK' if len(feat_list)==model_n else 'MISMATCH'})")
if scaler_n is not None:
    msg.append(f"Scaler expects:   {scaler_n}  ({'OK' if len(feat_list)==scaler_n else 'MISMATCH'})")
print("\n".join(msg))

# اگر mismatch داریم و پارکت فیچر موجوده، یک تلاش دیگر: ستون‌های مشترک با پارکت، به ترتیب پارکت
if (model_n is not None and len(feat_list) != model_n) and os.path.exists(FEATS_PQ):
    df = pd.read_parquet(FEATS_PQ, columns=None)
    cols = [c for c in df.columns if isinstance(c, str)]
    inter = [c for c in cols if c in set(feat_list)]
    if len(inter) > 0:
        feat_list = inter[:model_n] if model_n else inter

print(f"Final feature count: {len(feat_list)}")

# 4) ذخیره: بک‌آپ بگیر و سپس overwrite
backup = FEATURE_JS + ".bak"
if os.path.exists(FEATURE_JS):
    os.replace(FEATURE_JS, backup)
with open(FEATURE_JS, "w", encoding="utf-8") as f:
    json.dump(feat_list, f, ensure_ascii=False, indent=2)

print(f"✅ feature_list.json normalized and saved.\nBackup at: {backup if os.path.exists(backup) else '(no previous file)'}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Features resolved: 32
Model expects:    32  (OK)
Scaler expects:   32  (OK)
Final feature count: 32
✅ feature_list.json normalized and saved.
Backup at: /content/drive/MyDrive/colab_outputs/data/feature_list.json.bak


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/colab_outputs"
DATA_DIR = f"{SAVE_DIR}/data"

SCALER_IN   = f"{DATA_DIR}/robust_scaler.joblib"
MODEL_IN    = f"{DATA_DIR}/iforest_model.joblib"
FEATURE_JS  = f"{DATA_DIR}/feature_list.json"
META_JSON   = f"{DATA_DIR}/model_metadata.json"
OUT_CSV     = f"{SAVE_DIR}/inference_scored_with_severity.csv"

print("✅ Paths set. Go run your Cell B.")


Mounted at /content/drive
✅ Paths set. Go run your Cell B.
